# Calibración química y CO₂: partición principal y prueba inversa
**Principal: calibrar CO₂ con LAB019–021; holdout CO₂ LAB016–018.**
**Sensibilidad inversa: calibrar CO₂ con LAB016–018; holdout CO₂ LAB019–021.**

Las tres formulaciones reajustan únicamente sus coeficientes de CO₂ en cada partición, con la misma señal cruda reducida a medianas horarias, cobertura y ponderación por fermentación. La química sigue calibrada con LAB004–009. Sus parámetros guardados se reutilizan por defecto; activar REFIT_CHEMISTRY sólo para repetir ese ajuste.

El histórico conserva sus ecuaciones y theta_nv2: se reajustan kCO2_release_h, CO2sat_scale y matrix_gain para esta comparación. Los ajustes históricos anteriores se conservan en sus archivos originales; no se sustituyen. Los modelos nuevos ajustan kLa y gain. Misma inicialización por modelo en ambas particiones; ningún dato de holdout entra al objetivo.

Solubilidad de los modelos nuevos PROVISIONAL; Cd0 de LAB021 provisional y Cd0 de LAB016–018 asumido cero. Los resultados son ajustes locales y no validan estos supuestos. Sólo se guardan parámetros en configs/models, sin CSV ni PNG adicionales.

In [1]:
from pathlib import Path
import sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/sources.csv').exists())
sys.dont_write_bytecode=True
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
assert Path(sys.prefix).name.lower()=='fermentation'


In [2]:
from wine_fermentation.data import natural_batches
from wine_fermentation.shared.optimization import fit
REFIT_CHEMISTRY=False
fits={}
if REFIT_CHEMISTRY:
    _,batches,_=natural_batches(ROOT)
for name in ('without_transporters','with_transporters'):
    fits[name]=fit(ROOT,name,batches) if REFIT_CHEMISTRY else json.loads((ROOT/f'configs/models/{name}/fit.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{k:v for k,v in f.items() if k in ('model','success','initial_objective','final_objective','calls','seconds','message')} for f in fits.values()]))


,model,success,message,initial_objective,final_objective,calls,seconds
0,without_transporters,True,Both `ftol` and `xtol` termination conditions ...,12.293812,4.697006,269,56.797
1,with_transporters,True,Both `ftol` and `xtol` termination conditions ...,15.810357,4.470653,261,71.578


In [3]:
from wine_fermentation.comparison import run_both,PRIMARY_TRAINING
from IPython.display import Markdown
experiments=run_both(ROOT,fit_gas=True)
for key,res in experiments.items():
    display(Markdown('## '+('Principal' if key==PRIMARY_TRAINING else 'Inversa')+': '+key))
    display(pd.DataFrame(res['gas_fits']).T)
    display(res['metrics'])
summary=pd.concat([res['metrics'] for res in experiments.values()],ignore_index=True)
display(Markdown('## Comparación de CO₂: media de RMSE por fermentación [g/(L·h)]'))
display(summary.groupby(['model','training','split']).RMSE.mean().unstack(['training','split']))


historical train_019_021 objective 0.058697 success True


without_transporters train_019_021 objective 0.092432 success True


with_transporters train_019_021 objective 0.100425 success True


historical train_016_018 objective 0.019875 success True


without_transporters train_016_018 objective 0.06941 success True


...

## Inversa: train_016_018

,kCO2_release_h,CO2sat_scale,matrix_gain,success,message,training,training_labs,holdout_labs,objective,nfev,...,initial_parameters,parameter_bounds,target,weighting,chemical_parameters,historical_equations_unchanged,solubility_status,Cd0_unmeasured,kLa,gain
historical,0.175685,2.5,0.802053,True,`ftol` termination condition is satisfied.,train_016_018,"[LAB016, LAB017, LAB018]","[LAB019, LAB020, LAB021]",0.019875,6,...,"{'kCO2_release_h': 0.4, 'CO2sat_scale': 1.0, '...","{'kCO2_release_h': [0.03, 25.0], 'CO2sat_scale...",hourly_median_raw_g_l_h,equal_lab_mean_squared_error,theta_nv2,True,historical_unchanged,0 assumed for LAB016-018,NaN,NaN
without_transporters,NaN,NaN,NaN,True,`ftol` termination condition is satisfied.,train_016_018,"[LAB016, LAB017, LAB018]","[LAB019, LAB020, LAB021]",0.06941,13,...,"{'kLa': 1.0, 'gain': 0.7}","{'kLa': [0.01, 20.0], 'gain': [0.05, 3.0]}",hourly_median_raw_g_l_h,equal_lab_mean_squared_error,unchanged chemical fit.json,False,provisional_inherited_correlation,0 assumed for LAB016-018,0.221365,0.65456
with_transporters,NaN,NaN,NaN,True,`ftol` termination condition is satisfied.,train_016_018,"[LAB016, LAB017, LAB018]","[LAB019, LAB020, LAB021]",0.060315,16,...,"{'kLa': 1.0, 'gain': 0.7}","{'kLa': [0.01, 20.0], 'gain': [0.05, 3.0]}",hourly_median_raw_g_l_h,equal_lab_mean_squared_error,unchanged chemical fit.json,False,provisional_inherited_correlation,0 assumed for LAB016-018,0.431455,0.664575


,model,training,lab,n,split,RMSE,bias
0,historical,train_016_018,LAB016,152,gas_training,0.090438,0.003987
1,historical,train_016_018,LAB017,152,gas_training,0.077464,-0.010799
2,historical,train_016_018,LAB018,152,gas_training,0.075468,-0.008743
3,historical,train_016_018,LAB019,184,gas_validation,0.093162,0.006521
4,historical,train_016_018,LAB020,184,gas_validation,0.045171,-0.010291
5,historical,train_016_018,LAB021,184,gas_validation,0.242784,-0.041553
6,without_transporters,train_016_018,LAB016,152,gas_training,0.145655,0.013651
7,without_transporters,train_016_018,LAB017,152,gas_training,0.158045,-0.020615
8,without_transporters,train_016_018,LAB018,152,gas_training,0.152369,0.001421
9,without_transporters,train_016_018,LAB019,184,gas_validation,0.107860,0.026564


## Comparación de CO₂: media de RMSE por fermentación [g/(L·h)]

training             train_016_018                train_019_021               
split                 gas_training gas_validation  gas_training gas_validation
model                                                                         
historical                0.081123       0.127039      0.127064       0.120039
with_transporters         0.141776       0.168394      0.163169       0.147077
without_transporters      0.152023       0.165330      0.158207       0.157901